# Stage A — Synthetic EEW alignment and causality gate
The task is earthquake detection, then magnitude estimation, from observed early waveforms. Catalog features are **not** model inputs. This notebook verifies identity-based alignment, P/S annotations, splits and upstream signal causality before authorizing training.

Input: Phase 11 raw waveforms at 100 Hz. Proposed observations are [P,P+100), [P,P+300), [P,P+500) samples: 1/3/5 seconds post-P, no pre-arrival context. P anchors are oracle annotations; this is a triggered synthetic benchmark, not a deployed P picker or continuous false-alarm-rate test. Noise anchors use their explicitly paired earthquake's P index solely to match observation placement.

Do not crop full-trace standardized/detrended arrays: those transforms use samples beyond an early cutoff. Use raw waveforms, then fit any accepted normalization on training observations only. If any mandatory gate fails, candidate windows remain quarantined and downstream notebooks must skip training/evaluation.

In [1]:
from pathlib import Path
import json,gzip,hashlib,sys,time,importlib.util,dataclasses
import numpy as np
import pandas as pd
import h5py
ROOT=Path.cwd().resolve()
if ROOT.name!="bilstm new updated":ROOT=ROOT/"bilstm new updated"
PROJECT=ROOT.parent
WAVE=PROJECT/"cnn/cnn wave"
SIM=WAVE/"data wave/synthetic_waveform_simulator"
P11=SIM/"output/phase11_preprocessed_dataset"
for folder in ("dataset","results","models","reports",".agent-memory"): (ROOT/folder).mkdir(parents=True,exist_ok=True)
def digest(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for block in iter(lambda:f.read(4*1024*1024),b""):h.update(block)
    return h.hexdigest()
def save_json(path,obj):
    p=Path(path);tmp=p.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False),encoding="utf-8");tmp.replace(p)
audit=json.loads((WAVE/"phase1_waveform_cnn_results/reports/waveform_audit.json").read_text())
# This is the new project's source-only snapshot, not any earlier model dataset.
with gzip.open(ROOT/".cache/source_snapshot.json.gz","rt",encoding="utf-8") as f:snapshot=json.load(f)
assert digest(PROJECT/"dataset_2.csv")==snapshot["sha256"]
catalog=pd.DataFrame(snapshot["records"])
assert catalog.source_id.is_unique and catalog.trace_name.is_unique
catalog["origin_time_parsed"]=pd.to_datetime(catalog.origin_time_utc.str.replace(" UTC","",regex=False),format="mixed",utc=True,errors="raise")
catalog=catalog.sort_values("origin_time_parsed")
source_hashes={}
for relative,expected in audit["source_hashes"].items():
    p=WAVE/relative
    actual=digest(p)
    assert actual==expected,f"Changed source requires reassessment: {relative}"
    source_hashes[relative]=actual
source_files=[SIM/"phase11_split_and_preprocess.py",SIM/"phase10_generate_classification_dataset.py",
              SIM/"phase9_generate_full_dataset.py",SIM/"improved_waveform_generator.py",
              SIM/"output/phase9_full_dataset/simulation_parameters.json"]
for p in source_files:source_hashes[p.relative_to(WAVE).as_posix()]=digest(p)
RUN_ID=hashlib.sha256(json.dumps({"sources":source_hashes,"durations":[1,3,5],"version":1},sort_keys=True).encode()).hexdigest()[:16]
mapping=pd.read_csv(P11/"split_indices.csv",dtype={"source_id":"string","trace_name":"string"})
map10=pd.read_csv(SIM/"output/phase10_classification_dataset/earthquake_noise_mapping.csv",dtype={"source_id":"string","trace_name":"string"})
map9=pd.read_csv(SIM/"output/phase9_full_dataset/synthetic_waveforms_mapping.csv",dtype={"source_id":"string","trace_name":"string"})
arrival=pd.read_csv(SIM/"output/phase2_travel_time/metadata_with_theoretical_arrivals.csv",dtype={"source_id":"string","trace_name":"string"})
KEY=["source_id","trace_name"]
assert not mapping.duplicated(KEY).any() and not map10.duplicated(KEY).any()
assert not map9.duplicated(KEY).any() and not arrival.duplicated(KEY).any()
catalog_by_id=catalog.set_index("source_id")
frames=[]; schemas={}
for split in ("train","validation","test"):
    path=P11/f"{split}.h5"
    with h5py.File(path,"r") as h:
        schemas[split]={"shapes":{k:list(h[k].shape) for k in h},"sampling_hz":float(h.attrs["sampling_rate_hz"])}
        assert h["waveforms"].shape[1]==10000 and float(h.attrs["sampling_rate_hz"])==100.
        columns={}
        for k in ("source_id","trace_name","receiver_code","sample_type"):
            columns[k]=h[k].asstr()[:]
        for k in ("labels","pair_id","p_arrival_sample","s_arrival_sample","magnitude","depth_km","hypocentral_distance_km","snr_db","qc_pass"):
            columns[k]=h[k][:]
        df=pd.DataFrame(columns);df["h5_row"]=np.arange(len(df));df["partition"]=split
    aligned=df.merge(mapping,on=KEY,how="left",validate="one_to_one",suffixes=("","_manifest"),indicator=True)
    assert aligned._merge.eq("both").all()
    assert aligned["split"].eq(split).all()
    assert (aligned.labels==aligned.label).all() and (aligned.pair_id==aligned.pair_id_manifest).all()
    aligned=aligned.drop(columns="_merge")
    aligned=aligned.merge(map10[KEY+["combined_index","original_phase9_index","noise_profile_type"]],on=KEY,
                          how="left",validate="one_to_one",suffixes=("","_p10"),indicator=True)
    assert aligned._merge.eq("both").all() and (aligned.combined_index==aligned.combined_index_p10).all()
    aligned=aligned.drop(columns="_merge")
    eq=aligned.labels.eq(1)
    linked=aligned.loc[eq,KEY+["p_arrival_sample","s_arrival_sample","magnitude","original_phase9_index"]].merge(
        map9,on=KEY,how="left",validate="one_to_one",suffixes=("","_p9"),indicator=True)
    assert linked._merge.eq("both").all()
    assert (linked.original_phase9_index==linked.hdf5_index).all()
    assert (linked.p_arrival_sample==linked.p_arrival_sample_p9).all()
    assert (linked.s_arrival_sample==linked.s_arrival_sample_p9).all()
    assert np.allclose(linked.magnitude,linked.magnitude_p9,atol=1e-5)
    truth=aligned.loc[eq,KEY].merge(arrival,on=KEY,how="left",validate="one_to_one",indicator=True)
    assert truth._merge.eq("both").all()
    assert np.array_equal(aligned.loc[eq,"p_arrival_sample"],truth.p_arrival_sample)
    assert np.array_equal(aligned.loc[eq,"s_arrival_sample"],truth.s_arrival_sample)
    cat=aligned.loc[eq,KEY].merge(catalog,on=KEY,how="left",validate="one_to_one",indicator=True)
    assert cat._merge.eq("both").all()
    assert np.allclose(aligned.loc[eq,"magnitude"],pd.to_numeric(cat.Magnitude_Value),atol=1e-5)
    aligned["target_magnitude"]=np.where(eq,aligned.magnitude,np.nan)
    aligned["magnitude_type"]=None
    aligned["origin_time_utc_original"]=None
    aligned.loc[eq,"magnitude_type"]=cat.Magnitude_Type.to_numpy()
    aligned.loc[eq,"origin_time_utc_original"]=cat.origin_time_utc.to_numpy()
    counts=aligned.groupby("pair_id").labels.agg(["size","sum"])
    assert counts["size"].eq(2).all() and counts["sum"].eq(1).all()
    eqanchor=aligned.loc[eq].set_index("pair_id").p_arrival_sample
    aligned["anchor_sample"]=aligned.pair_id.map(eqanchor).astype(int)
    assert (aligned.loc[eq,"p_arrival_sample"]>=0).all()
    assert (aligned.loc[eq,"s_arrival_sample"]>aligned.loc[eq,"p_arrival_sample"]).all()
    for duration in (1,3,5):
        cutoff=aligned.anchor_sample+100*duration
        assert (cutoff<=10000).all()
        aligned[f"cutoff_sample_{duration}s"]=cutoff
        aligned[f"cutoff_before_s_{duration}s"]=eq & (cutoff<aligned.s_arrival_sample)
        aligned[f"contains_s_samples_{duration}s"]=eq & (cutoff>aligned.s_arrival_sample)
        aligned[f"synthetic_same_station_remaining_to_s_{duration}s"]=np.where(eq,(aligned.s_arrival_sample-cutoff)/100.,np.nan)
    frames.append(aligned)
metadata=pd.concat(frames,ignore_index=True)
assert metadata.trace_name.is_unique and metadata.source_id.is_unique
from itertools import combinations
for a,b in combinations(("train","validation","test"),2):
    assert set(metadata.loc[metadata.partition.eq(a),"pair_id"]).isdisjoint(metadata.loc[metadata.partition.eq(b),"pair_id"])
assert metadata.loc[metadata.labels.eq(0),"target_magnitude"].isna().all()
metadata.to_csv(ROOT/"dataset/event_waveform_manifest.csv",index=False)
save_json(ROOT/"dataset/source_manifest.json",{"run_id":RUN_ID,"source_hashes":source_hashes,"catalog_sha256":snapshot["sha256"],
    "waveform_key":"waveforms","identity_join":KEY,"sampling_hz":100,"schemas":schemas,
    "catalog_role":"trace identity, supervision and retrospective evaluation only",
    "catalog_real_time_availability":"UNVERIFIED; no publication/revision timestamps or invented delay",
    "split_protocol":"Preserved explicit Phase 11 event/noise pair-disjoint partitions for all future modalities."})
print("Identity joins, catalog magnitudes, P/S annotations and explicit pair-disjoint splits verified.")


Identity joins, catalog magnitudes, P/S annotations and explicit pair-disjoint splits verified.


## Upstream causal-noise test
A strictly sliced raw prefix is necessary but not sufficient if its generator used future information. The inspected simulator calibrates the ambient-noise scale from max(abs(clean_signal[P:end])). A later S-wave can therefore affect earlier noise. Test this on an identity-linked source event, keeping its P-wave, S arrival, catalog target and random seed fixed while changing only a later S-wave amplitude parameter. This is a diagnostic counterfactual in memory, not a new dataset or augmentation.

In [2]:
sys.dont_write_bytecode=True  # never create caches in the original simulator directory
spec=importlib.util.spec_from_file_location("_eew_source_generator",SIM/"improved_waveform_generator.py")
module=importlib.util.module_from_spec(spec);sys.modules[spec.name]=module;spec.loader.exec_module(module)
saved_params=json.loads((SIM/"output/phase9_full_dataset/simulation_parameters.json").read_text())
fields={f.name for f in dataclasses.fields(module.ImprovedSimulationParameters)}
params=module.ImprovedSimulationParameters(**{k:v for k,v in saved_params.items() if k in fields})
generator=module.ImprovedWaveformGenerator(params)
chosen=metadata[(metadata.labels==1)&(metadata.partition=="train")&
                (metadata.s_arrival_sample>metadata.anchor_sample+500)].iloc[0]
identity=(map9.source_id==chosen.source_id)&(map9.trace_name==chosen.trace_name)
entry=map9.loc[identity].iloc[0]
kwargs={"p_arrival_sample":int(entry.p_arrival_sample),"s_arrival_sample":int(entry.s_arrival_sample),
        "magnitude":float(entry.magnitude),"hypocentral_distance_km":float(entry.hypocentral_distance_km),
        "target_snr_db":float(entry.target_snr_db),"seed":int(entry.trace_seed)}
reference=generator.generate_synthesized_trace(**kwargs)
with h5py.File(P11/"train.h5","r") as h:
    stored=h["waveforms"][int(chosen.h5_row)]
reproduction_error=float(np.max(np.abs(reference["waveform"]-stored)))
# Float32 storage and platform math may produce small rounding differences.
reproduces=bool(np.allclose(reference["waveform"],stored,rtol=2e-5,atol=1e-7))
assert reproduces, "Current generator does not reproduce its keyed source trace; provenance requires review."
changed_params=dataclasses.replace(params,sp_amp_ratio_bhz=params.sp_amp_ratio_bhz*4.)
changed=module.ImprovedWaveformGenerator(changed_params).generate_synthesized_trace(**kwargs)
p=int(entry.p_arrival_sample);end=p+500
assert np.array_equal(reference["p_wave"],changed["p_wave"])
assert np.array_equal(reference["clean_signal"][:end],changed["clean_signal"][:end])
assert not np.array_equal(reference["waveform"][:end],changed["waveform"][:end])
causal_evidence={"source_id":str(entry.source_id),"trace_name":str(entry.trace_name),
    "source_trace_reproduced":reproduces,"source_max_abs_error":reproduction_error,
    "fixed":"P-wave, target magnitude, S arrival, distance, SNR and random seed",
    "counterfactual":"Multiply future S amplitude coefficient by four; no file changed",
    "same_clean_signal_before_5s_cutoff":True,"same_p_wave":True,
    "changed_observed_early_window":True,
    "early_window_max_abs_change":float(np.max(np.abs(reference["waveform"][p:end]-changed["waveform"][p:end]))),
    "pre_p_max_abs_change":float(np.max(np.abs(reference["waveform"][:p]-changed["waveform"][:p]))),
    "reference_noise_sigma":float(reference["target_noise_sigma"]),
    "changed_noise_sigma":float(changed["target_noise_sigma"]),
    "reference_full_signal_peak":float(reference["signal_peak"]),
    "changed_full_signal_peak":float(changed["signal_peak"]),
    "S_arrival_seconds":int(entry.s_arrival_sample)/100.,
    "latest_observation_cutoff_seconds":end/100.,
    "verdict":"FAIL: early ambient noise depends on unobserved future signal amplitude",
    "source_lines":"improved_waveform_generator.py:224-243"}
save_json(ROOT/"reports/causal_noise_counterexample.json",causal_evidence)
print(json.dumps(causal_evidence,indent=2))


{
  "source_id": "NCS_EV_07168",
  "trace_name": "ND.DBR..BHZ__earthquake_local__NCS_EV_07168",
  "source_trace_reproduced": true,
  "source_max_abs_error": 0.0,
  "fixed": "P-wave, target magnitude, S arrival, distance, SNR and random seed",
  "counterfactual": "Multiply future S amplitude coefficient by four; no file changed",
  "same_clean_signal_before_5s_cutoff": true,
  "same_p_wave": true,
  "changed_observed_early_window": true,
  "early_window_max_abs_change": 0.5218406915664673,
  "pre_p_max_abs_change": 0.5076061487197876,
  "reference_noise_sigma": 0.05420965796412107,
  "changed_noise_sigma": 0.2168358608698443,
  "reference_full_signal_peak": 1.3169723806810143,
  "changed_full_signal_peak": 5.267822204223828,
  "S_arrival_seconds": 33.86,
  "latest_observation_cutoff_seconds": 10.96,
  "verdict": "FAIL: early ambient noise depends on unobserved future signal amplitude",
  "source_lines": "improved_waveform_generator.py:224-243"
}


## Candidate crops — quarantined if causality fails
Save raw 1/3/5-second crops and explicit pre-S masks for inspection. Use identical slicing for earthquake/noise records. Noise has no magnitude target. No learned scaler is fitted while the gate is blocked. A candidate file is not permission to train: downstream notebooks must check reports/stage_a_validation.json.

Phase 11 full-parent duplicate checks are reused only after current source hashes match the audited files; candidate crops are additionally checked for exact cross-split duplicates. Within-parent duration variants remain in their original partition.

In [3]:
arrays={};duration_rows=[];crop_hashes={}
for split in ("train","validation","test"):
    df=metadata[metadata.partition==split].sort_values("h5_row").reset_index(drop=True)
    n=len(df)
    buffers={d:np.empty((n,d*100,1),dtype=np.float32) for d in (1,3,5)}
    with h5py.File(P11/f"{split}.h5","r") as h:
        for start in range(0,n,100):
            stop=min(start+100,n)
            max_end=int(df.iloc[start:stop].anchor_sample.max()+500)
            # Select only the range needed by the longest requested observation.
            block=h["waveforms"][start:stop,:max_end]
            for local,j in enumerate(range(start,stop)):
                anchor=int(df.anchor_sample.iloc[j])
                for duration in (1,3,5):
                    buffers[duration][j,:,0]=block[local,anchor:anchor+duration*100]
    arrays[f"label_{split}"]=df.labels.to_numpy(dtype=np.int64)
    arrays[f"magnitude_{split}"]=df.target_magnitude.to_numpy(dtype=np.float32)
    arrays[f"source_id_{split}"]=df.source_id.to_numpy(dtype=str)
    arrays[f"trace_name_{split}"]=df.trace_name.to_numpy(dtype=str)
    arrays[f"pair_id_{split}"]=df.pair_id.to_numpy(dtype=np.int64)
    for duration in (1,3,5):
        x=buffers[duration];assert np.isfinite(x).all()
        arrays[f"X_{split}_{duration}s"]=x
        mask=df[f"cutoff_before_s_{duration}s"].to_numpy(dtype=bool)
        arrays[f"pre_s_{split}_{duration}s"]=mask
        arrays[f"contains_s_{split}_{duration}s"]=df[f"contains_s_samples_{duration}s"].to_numpy(dtype=bool)
        eq=df.labels.eq(1)
        duration_rows.append({"partition":split,"duration_seconds":duration,"post_p_samples":duration*100,
            "shape":list(x.shape),"earthquakes":int(eq.sum()),"noise":int((~eq).sum()),
            "earthquake_cutoff_strictly_before_s_count":int(mask.sum()),
            "earthquake_cutoff_strictly_before_s_fraction":float(mask[eq].mean()),
            "earthquake_windows_containing_s_samples":int(df.loc[eq,f"contains_s_samples_{duration}s"].sum())})
        crop_hashes[(split,duration)]={hashlib.sha256(w.tobytes()).hexdigest() for w in x}
for duration in (1,3,5):
    for a,b in combinations(("train","validation","test"),2):
        assert crop_hashes[(a,duration)].isdisjoint(crop_hashes[(b,duration)])
arrays["run_id"]=np.array(RUN_ID)
arrays["training_allowed"]=np.array(False)
dest=ROOT/"dataset/eew_candidate_windows.npz"
with dest.with_suffix(".tmp").open("wb") as f:np.savez_compressed(f,**arrays)
dest.with_suffix(".tmp").replace(dest)
with np.load(dest,allow_pickle=False) as saved:
    assert all(np.array_equal(saved[k],v,equal_nan=True) if np.issubdtype(v.dtype,np.floating)
               else np.array_equal(saved[k],v) for k,v in arrays.items())
save_json(ROOT/"dataset/window_manifest.json",{"run_id":RUN_ID,"status":"QUARANTINED_CAUSALITY_FAILURE",
    "training_allowed":False,"sha256":digest(dest),"durations":duration_rows,
    "input":"one raw amplitude channel; post-P samples only; no magnitude/coordinates/simulator metadata inputs",
    "P_pick":"oracle annotation; noise uses pair-matched placement, not an actual noise P arrival",
    "cutoff_convention":"exclusive end P+100*duration; cutoff strictly before S requires end < S",
    "chronological_holdout":"NOT_BUILT while primary data causality gate fails. Catalog origins retained for a future separately frozen protocol."})
save_json(ROOT/"dataset/preprocessing_parameters.json",{
    "status":"NOT_FITTED_CAUSALITY_GATE_FAILED","waveform_key":"waveforms","sampling_hz":100,
    "pre_arrival_context_samples":0,"learned_scaling":None,"full_trace_standardization_used":False,
    "noise_magnitude_target":"NaN; excluded from any future magnitude loss",
    "augmentation_performed":False,"intended_training_only_comparison":"clean versus small amplitude perturbations and independently generated band-limited noise confined to each observed prefix; seed/config must be frozen after data repair"})
save_json(ROOT/"dataset/feature_definitions.json",{
    "channels":[{"index":0,"name":"raw_synthetic_vertical_amplitude","units":"simulator amplitude units"}],
    "sequence_axis":"100 Hz observed post-P signal samples, not catalog events",
    "excluded_inputs":["magnitude","latitude","longitude","source_id","trace_name","P/S metadata","simulator parameters"],
    "metadata_use":"identity matching, supervised labels, crop anchors, split and timing audits only"})
save_json(ROOT/"reports/window_summary.json",duration_rows)
print("Candidate raw windows saved; no scaler, augmentation, model training or tuning.")


Candidate raw windows saved; no scaler, augmentation, model training or tuning.


In [4]:
checks={
    "trace_identity_to_catalog_and_generator_manifest":"PASS",
    "P_S_annotation_identity_alignment":"PASS",
    "sampling_and_window_bounds":"PASS",
    "pair_event_and_noise_parent_split_integrity":"PASS",
    "full_parent_exact_duplicates":"PASS_REUSED_HASH_MATCHED_AUDIT",
    "candidate_exact_cross_split_duplicates":"PASS",
    "direct_model_input_ends_at_cutoff":"PASS",
    "full_trace_normalization_avoided":"PASS",
    "noise_magnitude_targets_masked":"PASS",
    "upstream_early_noise_independence_from_future_signal":"FAIL",
    "real_recorded_waveform_validation":"UNAVAILABLE",
    "oracle_P_pick_replaced_by_causal_detector":"NOT_IMPLEMENTED",
    "target_site_warning_arrival_validation":"UNAVAILABLE",
}
validation={"stage":"A","status":"BLOCKED","training_allowed":False,"checks":checks,
    "blocking_reason":"Verified source generator scales early ambient noise using the future full-trace clean-signal peak. A source-matched counterfactual changes future S amplitude and changes the observed pre-S prefix while its clean prefix remains identical.",
    "evidence":"causal_noise_counterexample.json","run_id":RUN_ID,
    "required_fix":"A separately versioned synthetic dataset whose ambient noise is sampled independently of target magnitude and future signal peaks, or aligned real recorded waveforms. Revalidate prefix invariance, identities, P/S and splits before training.",
    "current_candidate_action":"Preserved for review only; do not train.",
    "publication_time_policy":"Catalog origin ordering is retrospective only; reporting/revision availability UNVERIFIED; no assumed delay.",
    "preserved_original_sources":True}
save_json(ROOT/"reports/stage_a_validation.json",validation)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig,axes=plt.subplots(2,1,figsize=(10,6),sharex=True)
t=np.arange(p,end)/100.-p/100.
axes[0].plot(t,reference["clean_signal"][p:end],label="Original clean prefix")
axes[0].plot(t,changed["clean_signal"][p:end],"--",label="Future S amplitude changed")
axes[0].set(ylabel="Clean amplitude",title="Same clean 5-second post-P observation; S has not arrived")
axes[0].legend()
axes[1].plot(t,changed["waveform"][p:end]-reference["waveform"][p:end])
axes[1].set(xlabel="Seconds after annotated P",ylabel="Observed difference",
            title="Early noise changes because calibration uses the future signal peak")
fig.tight_layout();fig.savefig(ROOT/"reports/causal_noise_counterexample.png",dpi=140);plt.close(fig)
table=["| Split | Duration | Shape | Cutoff before S (earthquakes only) |","|---|---:|---|---:|"]
for r in duration_rows:
    table.append(f"| {r['partition']} | {r['duration_seconds']} s | {tuple(r['shape'])} | {r['earthquake_cutoff_strictly_before_s_fraction']:.2%} |")
report=f"""# BiLSTM EEW research report — Stage A blocked

## Outcome
Identity and split checks passed, but **the upstream causality gate failed**. No BiLSTM baseline, Optuna search, clean/augmented experiment or test evaluation was run. The saved waveform crops are quarantined, not an approved training dataset.

## Verified sources and labels
Phase 11 train/validation/test raw waveforms were verified against existing audit hashes, then joined to explicit split, Phase 10, Phase 9 and P/S manifests by **source_id + trace_name**. Catalog identities and magnitude labels also matched these keys. Pairs remain split-disjoint; full-parent duplicate checks were reused only after hash verification, and exact cross-split crop duplicates were independently excluded.
Catalog origin timestamps and IDs are preserved. Publication/revision times are absent; real-time catalog availability is UNVERIFIED and no reporting delay was invented. Catalog values are not model inputs.

## Candidate observations
100 Hz, 100-second single-channel synthetic parents. Extracted raw post-P windows of 100/300/500 samples; no pre-arrival context. No full-trace detrending/standardization is reused. Noise-only magnitudes are NaN, never zero. Magnitude targets preserve the original catalog's reported values/types; mixed-scale calibration would need explicit treatment before a magnitude experiment.

{chr(10).join(table)}

Windows with S motion are explicitly labeled; they cannot be represented as pre-S-only observations. Oracle P annotations anchor the windows, and noise uses explicitly paired anchors; this is not evidence of continuous detection or operational false alarms per hour.

## Confirmed blocker
improved_waveform_generator.py lines 224–243 sets background-noise scale using the maximum clean-signal amplitude over **P through the end of the full trace**.
For keyed event **{entry.source_id}**, the current generator reproduced the stored source trace (max absolute error {reproduction_error:.3g}). Keeping P, magnitude, S arrival, distance, SNR and seed fixed while multiplying only the later S-wave amplitude by four:
- Clean samples before the five-second cutoff stayed identical.
- Noise sigma changed from {reference['target_noise_sigma']:.6g} to {changed['target_noise_sigma']:.6g}.
- The observed early window changed by up to {causal_evidence['early_window_max_abs_change']:.6g}.
- S arrives at {int(entry.s_arrival_sample)/100:.2f} seconds in the trace, after the latest observation cutoff at {end/100:.2f} seconds.

Thus slicing early samples does not remove a future-signal-dependent synthetic noise cue. [Counterfactual evidence](causal_noise_counterexample.json) and [figure](causal_noise_counterexample.png) document the failure. It is a confirmed generator dependency, not a measured amount of model performance inflation.

## Training and evaluation status
Noise augmentation, fitted scalers, baseline/tuned hyperparameters, detection and magnitude metrics, processing latency and overfitting assessment are **NOT RUN / unavailable**, because Stage A failed. No checkpoint or Optuna study exists.
Observation durations are 1/3/5 seconds after an oracle P pick, not guaranteed warning times. Relative synthetic S-minus-cutoff intervals are retained, but no validated target-site arrival estimate exists; usable target-site warning lead time is UNKNOWN. A chronological holdout is deferred until causality is repaired.

## Research and next step
Graves & Schmidhuber supplies the forward/backward sequence-processing principle; a BiLSTM may use both directions only within the already observed prefix. The two earthquake-catalog papers are background, not evidence for P-wave EEW. See research_notes.md.
Create a separately versioned source with independent ambient noise (no target/future-peak calibration), or supply real recorded traces with verified P/S picks and identities. Re-run Stage A before any clean-versus-augmented BiLSTM training. Do not silently repair existing traces or reuse these quarantined crops.

All work is inside this new project. Previous data/models are unchanged; no CNN or hybrid code was modified or trained.
"""
(ROOT/"reports/final_research_report.md").write_text(report,encoding="utf-8")
save_json(ROOT/"reports/verification.json",{"stage_a_notebook_complete":True,"gate_enforced":True,
    "input_snapshot_hash_unchanged":digest(PROJECT/"dataset_2.csv")==snapshot["sha256"],
    "source_generator_files_unchanged":all(digest(x)==source_hashes[x.relative_to(WAVE).as_posix()] for x in source_files),
    "candidate_dataset_roundtrip_passed":True,"training_runs":0,"optuna_runs":0,
    "old_bilstm_artifacts_read":False,"source_data_opened_read_only":True})
print("Stage A BLOCKED: verified future-signal-dependent ambient noise. Report saved; training forbidden.")


Stage A BLOCKED: verified future-signal-dependent ambient noise. Report saved; training forbidden.
